# 05 m9 Pseudo-Load Iterative Search

This notebook is an exploratory, Beta-guided search for a stronger `m9_hybrid` variant using the physical idea:

```text
pseudo_load = solar_MW - net_load_MW
```

During wrong-sign RPF, this quantity may approximate underlying demand and become more stable inside the true RPF window. The notebook runs a bounded sequence of variants, records every result, and keeps all artifacts inside the ignored `99_Misc/outputs/` folder.

**Important.** This is not publication-ready validation. It trains only on Alpha, but it uses Beta results to guide/compare exploratory variants.

## 1. Imports, Paths, Controls

All writes are local to the misc output folder. The notebook reads the existing m9 candidate outputs from Notebook 03 so the overnight search can focus on pseudo-load feature and candidate variants rather than rebuilding the original m9 pipeline.

In [ ]:
from __future__ import annotations

import json
import time
from dataclasses import dataclass
from itertools import product
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import yaml

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from xgboost import XGBClassifier

RUN_FULL_SEARCH = True
RANDOM_SEED = 9
MAX_NEGATIVES_PER_DAY = 4
SEARCH_START_HOUR = 6
SEARCH_END_HOUR = 18
MIN_DURATION_MINUTES = 30
MAX_DURATION_MINUTES = 8 * 60
EPS = 1e-9
THRESHOLD_GRID = np.array([0.15, 0.20, 0.30, 0.45])

PALETTE = {
    "orange": "#eb932c",
    "dark_blue": "#22303d",
    "grey": "#2F4D67",
    "light_grey": "#5C7D99",
    "light_white": "#ebe3e3",
}
plt.rcParams.update({
    "font.family": "Arial",
    "axes.edgecolor": PALETTE["dark_blue"],
    "axes.labelcolor": PALETTE["dark_blue"],
    "axes.titlecolor": PALETTE["dark_blue"],
    "xtick.color": PALETTE["dark_blue"],
    "ytick.color": PALETTE["dark_blue"],
})


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "publication" / "2_journal_article" / "config" / "experiment_config.yaml").exists():
            return candidate
    raise FileNotFoundError("Could not find PyNRPF repo root.")


REPO_ROOT = find_repo_root()
ARTICLE_ROOT = REPO_ROOT / "publication" / "2_journal_article"
MISC_DIR = ARTICLE_ROOT / "notebooks" / "99_Misc"
M9_OUTPUT = MISC_DIR / "outputs" / "03_m9_hybrid_development"
V1B_OUTPUT = MISC_DIR / "outputs" / "04_m9_v1b_precision_gate_search"
OUTPUT_ROOT = MISC_DIR / "outputs" / "05_m9_pseudoload_iterative_search"
INTERMEDIATE_DIR = OUTPUT_ROOT / "intermediate"
METRICS_DIR = OUTPUT_ROOT / "metrics"
FIGURES_DIR = OUTPUT_ROOT / "figures"
HTML_DIR = OUTPUT_ROOT / "html"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [INTERMEDIATE_DIR, METRICS_DIR, FIGURES_DIR, HTML_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

with (ARTICLE_ROOT / "config" / "experiment_config.yaml").open("r", encoding="utf-8") as fh:
    CFG = yaml.safe_load(fh)

print("Output root:", OUTPUT_ROOT)
print("RUN_FULL_SEARCH:", RUN_FULL_SEARCH)


## 2. Loading, Metrics, And Decoding Helpers

These helpers are intentionally notebook-local. They load final datasets, existing m9 candidate outputs, compute binary metrics, and decode one selected candidate per site-day.

In [ ]:
EXPECTED_COLUMNS = ["substation_id", "date", "timestamp", "net_load_MW", "solar_MW", "label_interval", "label_day"]
COUNT_COLUMNS = ["support", "positive_support", "tp", "fp", "fn", "tn"]
SCORE_COLUMNS = ["precision", "recall", "f1"]


def write_csv(df: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return path


def load_final_dataset(dataset_key: str) -> pd.DataFrame:
    path = ARTICLE_ROOT / CFG["paths"][f"{dataset_key}_dataset_path"]
    df = pd.read_parquet(path)[EXPECTED_COLUMNS].copy()
    ts = pd.to_datetime(df["timestamp"], errors="coerce")
    if getattr(ts.dt, "tz", None) is not None:
        ts = ts.dt.tz_localize(None)
    df["timestamp"] = ts
    df["date"] = df["date"].astype(str)
    df["label_interval"] = df["label_interval"].astype(bool)
    df["label_day"] = df.groupby(["substation_id", "date"])["label_interval"].transform("any")
    return df.sort_values(["substation_id", "timestamp"]).reset_index(drop=True)


def read_csv_dates(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    for col in ["pred_start", "pred_end", "true_start", "true_end", "left_min_time", "right_min_time", "peak_time", "solar_peak_time"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    if "date" in df.columns:
        df["date"] = df["date"].astype(str)
    return df


def binary_metrics(y_true: Iterable[Any], y_pred: Iterable[Any]) -> dict[str, Any]:
    true = np.asarray(list(y_true), dtype=bool)
    pred = np.asarray(list(y_pred), dtype=bool)
    tp = int((true & pred).sum())
    fp = int((~true & pred).sum())
    fn = int((true & ~pred).sum())
    tn = int((~true & ~pred).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"support": len(true), "positive_support": int(true.sum()), "tp": tp, "fp": fp, "fn": fn, "tn": tn, "precision": precision, "recall": recall, "f1": f1}


def true_windows(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (site, date), grp in df.groupby(["substation_id", "date"], sort=False):
        labelled = grp.loc[grp["label_interval"]]
        rows.append({
            "substation_id": site,
            "date": date,
            "label_day": not labelled.empty,
            "true_start": labelled["timestamp"].iloc[0] if not labelled.empty else pd.NaT,
            "true_end": labelled["timestamp"].iloc[-1] if not labelled.empty else pd.NaT,
        })
    return pd.DataFrame(rows)


def decode_days(eval_df: pd.DataFrame, scored: pd.DataFrame, score_col: str, gate: dict[str, Any]) -> tuple[pd.DataFrame, pd.DataFrame]:
    days = true_windows(eval_df)
    cand = scored.copy()
    if not cand.empty:
        mask = cand[score_col] >= gate["threshold"]
        for col, op, value in gate.get("filters", []):
            if op == ">=":
                mask &= cand[col] >= value
            elif op == "<=":
                mask &= cand[col] <= value
            elif op == "abs<=":
                mask &= cand[col].abs() <= value
        cand = cand.loc[mask].copy()
    if cand.empty:
        best = pd.DataFrame(columns=["substation_id", "date", "candidate_id", "pred_start", "pred_end", score_col])
    else:
        best = (
            cand.sort_values(["substation_id", "date", score_col], ascending=[True, True, False])
            .groupby(["substation_id", "date"], as_index=False)
            .head(1)[["substation_id", "date", "candidate_id", "pred_start", "pred_end", score_col]]
        )
    decoded = days.merge(best, on=["substation_id", "date"], how="left")
    decoded[score_col] = decoded[score_col].fillna(0.0)
    decoded["pred_day"] = decoded[score_col] >= gate["threshold"]
    decoded.loc[~decoded["pred_day"], ["pred_start", "pred_end"]] = pd.NaT
    interval = eval_df.merge(decoded[["substation_id", "date", "pred_day", "pred_start", "pred_end"]], on=["substation_id", "date"], how="left")
    interval["pred_interval"] = (
        interval["pred_day"].fillna(False)
        & (interval["timestamp"] >= interval["pred_start"])
        & (interval["timestamp"] <= interval["pred_end"])
    )
    interval["hour"] = interval["timestamp"].dt.hour
    return decoded, interval


def metric_rows(eval_df: pd.DataFrame, scored: pd.DataFrame, dataset: str, fold_id: str, score_col: str, gate: dict[str, Any]) -> pd.DataFrame:
    decoded, interval = decode_days(eval_df, scored, score_col, gate)
    rows = [{"dataset": dataset, "fold_id": fold_id, "level": "day", **binary_metrics(decoded["label_day"], decoded["pred_day"])}]
    daytime = interval["hour"].between(SEARCH_START_HOUR, SEARCH_END_HOUR, inclusive="both")
    rows.append({"dataset": dataset, "fold_id": fold_id, "level": "interval", **binary_metrics(interval.loc[daytime, "label_interval"], interval.loc[daytime, "pred_interval"])})
    return pd.DataFrame(rows)


## 3. Pseudo-Load Features And Candidate Expansions

The pseudo-load feature builder works on any candidate table with `pred_start` and `pred_end`. Expanded candidate variants are created locally from the existing m9 candidates; no main workflow code is touched.

In [ ]:
def finite_percentile(values: np.ndarray, q: float, default: float = 0.0) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return default if len(values) == 0 else float(np.nanpercentile(values, q))


def roughness(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return 0.0 if len(values) < 2 else float(np.abs(np.diff(values)).sum())


def mean_abs_slope(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return 0.0 if len(values) < 2 else float(np.abs(np.diff(values)).mean())


def linear_slope(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    mask = np.isfinite(values)
    if mask.sum() < 2:
        return 0.0
    x = np.arange(len(values))[mask]
    y = values[mask]
    return float(np.polyfit(x, y, 1)[0])


def safe_ratio(num: float, den: float) -> float:
    return float(num / max(abs(den), EPS))



def add_pseudoload_features(raw: pd.DataFrame, candidates: pd.DataFrame) -> pd.DataFrame:
    """Add pseudo-load features with O(site-day + candidate) keyed lookup.

    The first implementation filtered the full candidate table inside every
    site-day loop. Boundary-expanded candidates make that pattern painfully
    slow, so this version pre-groups candidates by site-day once.
    """
    if candidates.empty:
        return candidates.copy()
    cand = candidates.copy()
    for col in ["pred_start", "pred_end"]:
        cand[col] = pd.to_datetime(cand[col], errors="coerce")
    cand["_date_key"] = cand["date"].astype(str)
    grouped_candidates = {
        key: group.drop(columns=["_date_key"]).copy()
        for key, group in cand.groupby(["substation_id", "_date_key"], sort=False)
    }

    rows = []
    for (site, date), grp in raw.groupby(["substation_id", "date"], sort=False):
        day_cand = grouped_candidates.get((site, str(date)))
        if day_cand is None or day_cand.empty:
            continue
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        ts = grp["timestamp"]
        net = grp["net_load_MW"].to_numpy(dtype=float)
        solar = grp["solar_MW"].to_numpy(dtype=float)
        pseudo = solar - net
        daytime = ts.dt.hour.between(SEARCH_START_HOUR, SEARCH_END_HOUR, inclusive="both").to_numpy()
        daytime_pseudo_rough = roughness(pseudo[daytime])
        daytime_pseudo_range = finite_percentile(pseudo[daytime], 100) - finite_percentile(pseudo[daytime], 0)

        ts_values = ts.to_numpy()
        for _, row in day_cand.iterrows():
            start, end = pd.Timestamp(row["pred_start"]), pd.Timestamp(row["pred_end"])
            mask = (ts_values >= np.datetime64(start)) & (ts_values <= np.datetime64(end))
            idx = np.where(mask)[0]
            if len(idx) == 0:
                continue
            p = pseudo[idx]
            s = solar[idx]
            n = net[idx]
            pseudo_rough = roughness(p)
            pseudo_range = finite_percentile(p, 100) - finite_percentile(p, 0)
            out = row.to_dict()
            out.update({
                "pseudo_load_mean_inside": finite_percentile(p, 50),
                "pseudo_load_std_inside": float(np.nanstd(p)) if np.isfinite(p).any() else 0.0,
                "pseudo_load_range_inside": pseudo_range,
                "pseudo_load_roughness_inside": pseudo_rough,
                "pseudo_load_mean_abs_slope_inside": mean_abs_slope(p),
                "pseudo_load_linear_slope_inside": linear_slope(p),
                "pseudo_roughness_vs_solar": safe_ratio(pseudo_rough, roughness(s)),
                "pseudo_roughness_vs_net": safe_ratio(pseudo_rough, roughness(n)),
                "pseudo_roughness_vs_daytime": safe_ratio(pseudo_rough, daytime_pseudo_rough),
                "pseudo_range_vs_daytime": safe_ratio(pseudo_range, daytime_pseudo_range),
                "pseudo_stability_score": 1.0 / (1.0 + safe_ratio(pseudo_rough, daytime_pseudo_rough) + safe_ratio(pseudo_range, daytime_pseudo_range)),
            })
            rows.append(out)
    return pd.DataFrame(rows)

def relabel_alpha_candidates(candidates: pd.DataFrame, raw: pd.DataFrame) -> pd.DataFrame:
    windows = true_windows(raw)
    out = candidates.drop(columns=["label_day", "true_start", "true_end", "start_error_minutes", "end_error_minutes", "iou_with_true", "is_positive"], errors="ignore").merge(windows, on=["substation_id", "date"], how="left")
    starts, ends, ious, positives = [], [], [], []
    for _, row in out.iterrows():
        if not bool(row["label_day"]) or pd.isna(row["true_start"]) or pd.isna(row["true_end"]):
            starts.append(np.nan); ends.append(np.nan); ious.append(0.0); positives.append(False); continue
        start_error = abs((pd.Timestamp(row["pred_start"]) - pd.Timestamp(row["true_start"])).total_seconds()) / 60
        end_error = abs((pd.Timestamp(row["pred_end"]) - pd.Timestamp(row["true_end"])).total_seconds()) / 60
        a0, a1 = pd.Timestamp(row["pred_start"]), pd.Timestamp(row["pred_end"]) + pd.Timedelta(minutes=15)
        b0, b1 = pd.Timestamp(row["true_start"]), pd.Timestamp(row["true_end"]) + pd.Timedelta(minutes=15)
        overlap = max(pd.Timedelta(0), min(a1, b1) - max(a0, b0)).total_seconds() / 60
        union = (max(a1, b1) - min(a0, b0)).total_seconds() / 60
        starts.append(start_error); ends.append(end_error); ious.append(overlap / union if union else 0.0)
        positives.append(start_error <= 30 and end_error <= 30)
    out["start_error_minutes"] = starts
    out["end_error_minutes"] = ends
    out["iou_with_true"] = ious
    out["is_positive"] = positives
    return out


def basic_candidate_columns(df: pd.DataFrame, source: str) -> pd.DataFrame:
    cols = ["substation_id", "date", "candidate_id", "pred_start", "pred_end"]
    out = df[cols].copy()
    out["candidate_source"] = source
    return out.drop_duplicates(["substation_id", "date", "pred_start", "pred_end", "candidate_source"]).reset_index(drop=True)


def boundary_expand(base: pd.DataFrame) -> pd.DataFrame:
    shifts = [-15, 0, 15]
    rows = []
    base_small = basic_candidate_columns(base, "boundary_expanded")
    for _, row in base_small.iterrows():
        original_start = pd.Timestamp(row["pred_start"])
        original_end = pd.Timestamp(row["pred_end"])
        day = original_start.normalize()
        lo = day + pd.Timedelta(hours=SEARCH_START_HOUR)
        hi = day + pd.Timedelta(hours=SEARCH_END_HOUR)
        for ss, ee in product(shifts, shifts):
            start = original_start + pd.Timedelta(minutes=ss)
            end = original_end + pd.Timedelta(minutes=ee)
            duration = (end - start).total_seconds() / 60 + 15
            if start < lo or end > hi or duration < MIN_DURATION_MINUTES or duration > MAX_DURATION_MINUTES or start > end:
                continue
            r = row.to_dict()
            r["pred_start"] = start
            r["pred_end"] = end
            rows.append(r)
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out = out.drop_duplicates(["substation_id", "date", "pred_start", "pred_end"]).reset_index(drop=True)
    out["candidate_id"] = out.groupby(["substation_id", "date"]).cumcount()
    return out


def pseudoload_segment_candidates(raw: pd.DataFrame, top_k: int = 2) -> pd.DataFrame:
    durations = [4, 8, 12, 16]  # 1h, 2h, 3h, 4h in 15-min intervals
    rows = []
    for (site, date), grp in raw.groupby(["substation_id", "date"], sort=False):
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        ts = grp["timestamp"]
        net = grp["net_load_MW"].to_numpy(dtype=float)
        solar = grp["solar_MW"].to_numpy(dtype=float)
        pseudo = solar - net
        day_p95_solar = finite_percentile(solar[ts.dt.hour.between(SEARCH_START_HOUR, SEARCH_END_HOUR, inclusive="both")], 95)
        candidates = []
        for dur in durations:
            for start_idx in range(len(grp) - dur + 1):
                end_idx = start_idx + dur - 1
                start, end = ts.iloc[start_idx], ts.iloc[end_idx]
                if start.hour < SEARCH_START_HOUR or end.hour > SEARCH_END_HOUR:
                    continue
                s = solar[start_idx : end_idx + 1]
                if finite_percentile(s, 95) < max(0.05 * day_p95_solar, EPS):
                    continue
                p = pseudo[start_idx : end_idx + 1]
                score = roughness(p) + 0.25 * (finite_percentile(p, 100) - finite_percentile(p, 0))
                candidates.append((score, start, end))
        for cid, (_, start, end) in enumerate(sorted(candidates, key=lambda x: x[0])[:top_k]):
            rows.append({"substation_id": site, "date": str(date), "candidate_id": cid, "pred_start": start, "pred_end": end, "candidate_source": "pseudoload_segment"})
    return pd.DataFrame(rows)


## 4. Training, Scoring, And Variant Evaluation

Each trainable variant uses Alpha only. Beta metrics are then used to compare and guide this exploratory search. The variant list is intentionally bounded to 12 entries.

In [ ]:
BASE_FEATURES = [
    "start_hour", "end_hour", "midpoint_hour", "duration_hours", "weekend_flag",
    "solar_p95_inside", "solar_peak_inside", "solar_bell_score",
    "net_load_p05_inside", "net_load_p95_inside", "net_load_range_inside", "net_load_peak_positive_inside",
    "net_load_norm_p05_inside", "net_load_norm_p95_inside", "net_load_n_shape_score",
    "solar_net_corr", "derivative_same_sign_fraction", "mean_derivative_product", "ramp_up_comovement", "ramp_down_comovement",
    "contains_daily_solar_peak", "bounce_height_MW", "distance_midpoint_to_solar_peak_minutes",
    "missing_intervals_inside", "missing_net_load_day", "missing_solar_day",
]
PSEUDO_FEATURES = [
    "pseudo_load_mean_inside", "pseudo_load_std_inside", "pseudo_load_range_inside",
    "pseudo_load_roughness_inside", "pseudo_load_mean_abs_slope_inside", "pseudo_load_linear_slope_inside",
    "pseudo_roughness_vs_solar", "pseudo_roughness_vs_net", "pseudo_roughness_vs_daytime",
    "pseudo_range_vs_daytime", "pseudo_stability_score",
]
CATEGORICAL_FEATURES = ["month", "weekday", "candidate_source"]


@dataclass
class VariantSpec:
    variant_id: str
    description: str
    candidate_set: str
    feature_mode: str = "pseudo"
    selection_mode: str = "max_f1"
    gate_mode: str = "none"
    rerank_weight: float = 0.0
    hard_negative_mode: str = "default"
    xgb_depth: int = 3
    xgb_estimators: int = 70
    beta_guided_note: str = ""


def feature_columns_for(mode: str) -> tuple[list[str], list[str]]:
    numeric = list(BASE_FEATURES)
    if mode in {"pseudo", "pseudo_only"}:
        numeric = (BASE_FEATURES if mode == "pseudo" else []) + PSEUDO_FEATURES
    return numeric, CATEGORICAL_FEATURES


def make_feature_matrix(df: pd.DataFrame, mode: str, columns: list[str] | None = None) -> tuple[pd.DataFrame, list[str]]:
    numeric, categorical = feature_columns_for(mode)
    available_cats = [c for c in categorical if c in df.columns]
    base = df.reindex(columns=numeric + available_cats, fill_value=0.0).copy()
    base = pd.get_dummies(base, columns=available_cats, prefix=available_cats, dtype=float)
    base = base.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if columns is not None:
        base = base.reindex(columns=columns, fill_value=0.0)
        return base, columns
    return base, base.columns.tolist()


def sample_train_rows(labelled: pd.DataFrame, mode: str, seed: int = RANDOM_SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    parts = []
    for _, grp in labelled.groupby(["substation_id", "date"], sort=False):
        pos = grp.loc[grp["is_positive"]]
        neg = grp.loc[~grp["is_positive"]].copy()
        if not pos.empty:
            parts.append(pos)
        if neg.empty:
            continue
        neg["_iou"] = neg["iou_with_true"].fillna(0.0)
        neg["_boundary"] = neg[["start_error_minutes", "end_error_minutes"]].fillna(10_000).sum(axis=1)
        neg["_solar"] = neg["solar_peak_inside"].fillna(0.0) if "solar_peak_inside" in neg.columns else 0.0
        neg["_pseudo_hard"] = neg["pseudo_stability_score"].fillna(0.0) if "pseudo_stability_score" in neg.columns else 0.0
        neg["_random"] = rng.random(len(neg))
        if mode == "pseudo_hard":
            sort_cols = ["_pseudo_hard", "_iou", "_boundary", "_solar", "_random"]
            ascending = [False, False, True, False, True]
        else:
            sort_cols = ["_iou", "_boundary", "_solar", "_random"]
            ascending = [False, True, False, True]
        parts.append(neg.sort_values(sort_cols, ascending=ascending).head(MAX_NEGATIVES_PER_DAY).drop(columns=["_iou", "_boundary", "_solar", "_pseudo_hard", "_random"]))
    return pd.concat(parts, ignore_index=True) if parts else labelled.iloc[0:0].copy()


def train_model(labelled: pd.DataFrame, spec: VariantSpec, seed: int) -> tuple[XGBClassifier, list[str]]:
    train = sample_train_rows(labelled, spec.hard_negative_mode, seed)
    if train.empty or train["is_positive"].nunique() < 2:
        raise ValueError(f"{spec.variant_id}: training rows need positive and negative candidates.")
    X, columns = make_feature_matrix(train, spec.feature_mode)
    y = train["is_positive"].astype(int).to_numpy()
    pos = max(1, int(y.sum()))
    neg = max(1, int(len(y) - y.sum()))
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_estimators=spec.xgb_estimators,
        max_depth=spec.xgb_depth,
        learning_rate=0.07,
        subsample=0.85,
        colsample_bytree=0.85,
        scale_pos_weight=neg / pos,
        random_state=seed,
        n_jobs=4,
    )
    model.fit(X, y)
    return model, columns


def score_with_model(model: XGBClassifier, columns: list[str], features: pd.DataFrame, spec: VariantSpec) -> pd.DataFrame:
    scored = features.copy()
    X, _ = make_feature_matrix(scored, spec.feature_mode, columns)
    scored["candidate_probability"] = model.predict_proba(X)[:, 1]
    if spec.rerank_weight:
        scored["candidate_score"] = scored["candidate_probability"] + spec.rerank_weight * scored["pseudo_stability_score"].fillna(0.0)
    else:
        scored["candidate_score"] = scored["candidate_probability"]
    return scored


def gate_filters_for(mode: str, row: pd.Series | None = None) -> list[tuple[str, str, float]]:
    if mode == "none":
        return []
    if row is None:
        raise ValueError("row is required for gate filters")
    filters = []
    if mode in {"pseudo_gate", "combined_gate"}:
        filters.extend([
            ("pseudo_roughness_vs_daytime", "<=", float(row["max_pseudo_rough_day"])),
            ("pseudo_range_vs_daytime", "<=", float(row["max_pseudo_range_day"])),
        ])
    if mode in {"v1b_gate", "combined_gate"}:
        filters.extend([
            ("solar_p95_inside", ">=", float(row["min_solar_p95"])),
            ("derivative_same_sign_fraction", ">=", float(row["min_same_sign"])),
            ("solar_net_corr", ">=", float(row["min_corr"])),
            ("duration_hours", "<=", float(row["max_duration_hours"])),
        ])
    return filters


def gate_grid(mode: str) -> pd.DataFrame:
    rows = []
    if mode == "none":
        return pd.DataFrame({"threshold": THRESHOLD_GRID})
    # Deliberately small grid: enough to test pseudo-load gates without making
    # each variant decode hundreds of times.
    pseudo_grid = [999.0, 0.75] if mode in {"pseudo_gate", "combined_gate"} else [999.0]
    range_grid = [999.0] if mode in {"pseudo_gate", "combined_gate"} else [999.0]
    solar_grid = [0.0, 2.0] if mode in {"v1b_gate", "combined_gate"} else [0.0]
    same_grid = [0.0]
    corr_grid = [-1.0]
    dur_grid = [8.0]
    for threshold, pr, rg, solar, same, corr, dur in product(THRESHOLD_GRID, pseudo_grid, range_grid, solar_grid, same_grid, corr_grid, dur_grid):
        rows.append({
            "threshold": float(threshold),
            "max_pseudo_rough_day": float(pr),
            "max_pseudo_range_day": float(rg),
            "min_solar_p95": float(solar),
            "min_same_sign": float(same),
            "min_corr": float(corr),
            "max_duration_hours": float(dur),
        })
    return pd.DataFrame(rows)


def select_gate(alpha_raw: pd.DataFrame, alpha_scored: pd.DataFrame, spec: VariantSpec) -> tuple[dict[str, Any], pd.DataFrame]:
    rows = []
    for _, gate_row in gate_grid(spec.gate_mode).iterrows():
        gate = {"threshold": float(gate_row["threshold"]), "filters": gate_filters_for(spec.gate_mode, gate_row)}
        day = metric_rows(alpha_raw, alpha_scored, "Alpha", "alpha_oof", "candidate_score", gate).loc[lambda d: d["level"].eq("day")].iloc[0]
        rows.append({**gate_row.to_dict(), **{f"alpha_day_{c}": day[c] for c in COUNT_COLUMNS + SCORE_COLUMNS}})
    sweep = pd.DataFrame(rows)
    if spec.selection_mode.startswith("precision_floor_"):
        floor = float(spec.selection_mode.split("_")[-1])
        eligible = sweep.loc[sweep["alpha_day_precision"] >= floor]
        pool = eligible if not eligible.empty else sweep
        sort_cols = ["alpha_day_f1", "alpha_day_precision", "alpha_day_recall"]
    else:
        pool = sweep
        sort_cols = ["alpha_day_f1", "alpha_day_precision", "alpha_day_recall"]
    best = pool.sort_values(sort_cols, ascending=[False, False, False]).iloc[0]
    gate = {"threshold": float(best["threshold"]), "filters": gate_filters_for(spec.gate_mode, best)}
    return gate, sweep


## 5. Load Data And Prepare Candidate Sets

The base m7 candidate rows come from the previous full m9 notebook. Boundary-expanded and pseudo-load segment candidates are generated only inside this notebook.

In [ ]:
t0 = time.perf_counter()
alpha = load_final_dataset("alpha")
beta = load_final_dataset("beta")

alpha_base_labelled = read_csv_dates(M9_OUTPUT / "intermediate" / "02_alpha_candidate_features_labels.csv")
alpha_base_scored = read_csv_dates(M9_OUTPUT / "intermediate" / "04_alpha_loso_scored_candidates.csv")
beta_base_scored = read_csv_dates(M9_OUTPUT / "intermediate" / "06_beta_scored_candidates.csv")

alpha_base = add_pseudoload_features(alpha, alpha_base_labelled.assign(candidate_source="m7"))
alpha_base = relabel_alpha_candidates(alpha_base, alpha)
beta_base = add_pseudoload_features(beta, beta_base_scored.assign(candidate_source="m7"))

alpha_boundary = add_pseudoload_features(alpha, boundary_expand(alpha_base_labelled))
alpha_boundary = relabel_alpha_candidates(alpha_boundary, alpha)
beta_boundary = add_pseudoload_features(beta, boundary_expand(beta_base_scored))

alpha_segments = add_pseudoload_features(alpha, pseudoload_segment_candidates(alpha))
alpha_segments = relabel_alpha_candidates(alpha_segments, alpha)
beta_segments = add_pseudoload_features(beta, pseudoload_segment_candidates(beta))

def combine_candidates(*frames: pd.DataFrame) -> pd.DataFrame:
    out = pd.concat([f for f in frames if f is not None and not f.empty], ignore_index=True, sort=False)
    if out.empty:
        return out
    out = out.drop_duplicates(["substation_id", "date", "pred_start", "pred_end", "candidate_source"]).reset_index(drop=True)
    out["candidate_id"] = out.groupby(["substation_id", "date"]).cumcount()
    return out

candidate_sets = {
    "m7": (alpha_base, beta_base),
    "boundary": (alpha_boundary, beta_boundary),
    "segments": (alpha_segments, beta_segments),
    "m7_plus_segments": (combine_candidates(alpha_base, alpha_segments), combine_candidates(beta_base, beta_segments)),
    "all": (combine_candidates(alpha_base, alpha_boundary, alpha_segments), combine_candidates(beta_base, beta_boundary, beta_segments)),
}

candidate_summary = pd.DataFrame([
    {"candidate_set": name, "alpha_rows": len(a), "beta_rows": len(b), "alpha_positive_rows": int(a.get("is_positive", pd.Series(dtype=bool)).sum())}
    for name, (a, b) in candidate_sets.items()
])
write_csv(candidate_summary, INTERMEDIATE_DIR / "00_candidate_set_summary.csv")
display(candidate_summary)


## 6. Run Baselines And 10 Pseudo-Load Variants

The sequence starts from current m9/v1b baselines, then tries pseudo-load feature, gate, reranking, hard-negative, boundary-expanded, pseudo-segment, combined, and small hyperparameter variants.

In [ ]:
variant_specs = [
    VariantSpec("v02_pseudo_xgb_m7", "Pseudo-load features on original m7 candidates.", "m7", "pseudo", "max_f1", "none"),
    VariantSpec("v03_pseudo_xgb_m7_precision90", "Pseudo features with Alpha precision floor 0.90.", "m7", "pseudo", "precision_floor_0.9", "none"),
    VariantSpec("v04_pseudo_gate_m7", "Pseudo features with pseudo-load stability gate.", "m7", "pseudo", "max_f1", "pseudo_gate"),
    VariantSpec("v05_combined_gate_m7", "Pseudo features with pseudo + v1b style gates.", "m7", "pseudo", "max_f1", "combined_gate"),
    VariantSpec("v06_pseudo_rerank_m7", "Rerank by XGB probability plus pseudo stability.", "m7", "pseudo", "max_f1", "none", rerank_weight=0.18),
    VariantSpec("v07_pseudo_hard_negative_m7", "Pseudo features with pseudo-stable hard negatives.", "m7", "pseudo", "max_f1", "none", hard_negative_mode="pseudo_hard"),
    VariantSpec("v08_boundary_expanded_xgb", "Boundary-expanded candidates with pseudo-only features.", "boundary", "pseudo_only", "max_f1", "none"),
    VariantSpec("v09_pseudoload_segments_xgb", "Pseudo-load constant-segment candidates only.", "segments", "pseudo_only", "max_f1", "none"),
    VariantSpec("v10_m7_plus_segments_xgb", "Original m7 candidates plus pseudo-load segment candidates.", "m7_plus_segments", "pseudo_only", "max_f1", "none"),
    VariantSpec("v11_pseudo_only_m7", "Pseudo-load-only feature ablation on original m7 candidates.", "m7", "pseudo_only", "max_f1", "none"),
    VariantSpec("v12_combined_gate_depth4", "Combined gates with a modestly deeper pseudo-feature XGB.", "m7", "pseudo", "max_f1", "combined_gate", xgb_depth=4, xgb_estimators=90),
]

baseline_rows = []

# Current m9 baseline: existing OOF Alpha threshold selection, existing Beta scores.
alpha_existing = add_pseudoload_features(alpha, alpha_base_scored.assign(candidate_source="m7"))
alpha_existing["candidate_score"] = alpha_existing["candidate_probability"]
beta_existing = add_pseudoload_features(beta, beta_base_scored.assign(candidate_source="m7"))
beta_existing["candidate_score"] = beta_existing["candidate_probability"]
gate, sweep = select_gate(alpha, alpha_existing, VariantSpec("baseline", "Current m9", "m7", "pseudo", "max_f1", "none"))
for _, row in metric_rows(beta, beta_existing, "Beta", "current_m9_existing", "candidate_score", gate).iterrows():
    baseline_rows.append({"variant_id": "current_m9_existing", "description": "Existing m9 scored candidates", "selected_gate": json.dumps(gate), **row.to_dict()})

# Existing v1b-inspired fixed gate.
v1b_gate = {"threshold": 0.2, "filters": [("solar_p95_inside", ">=", 2.0), ("duration_hours", "<=", 8.0)]}
for _, row in metric_rows(beta, beta_existing, "Beta", "v1b_existing_gate", "candidate_score", v1b_gate).iterrows():
    baseline_rows.append({"variant_id": "v1b_existing_gate", "description": "Existing v1b precision gate", "selected_gate": json.dumps(v1b_gate), **row.to_dict()})

baseline_metrics = pd.DataFrame(baseline_rows)
write_csv(baseline_metrics, METRICS_DIR / "01_baseline_metrics.csv")

variant_metric_rows = []
variant_detail_rows = []
beta_site_rows = []
error_summary_rows = []
all_sweeps = []
scored_cache: dict[str, pd.DataFrame] = {}

for idx, spec in enumerate(variant_specs, start=1):
    print(f"[{idx}/{len(variant_specs)}] {spec.variant_id}: {spec.description}")
    alpha_features, beta_features = candidate_sets[spec.candidate_set]
    model, columns = train_model(alpha_features, spec, RANDOM_SEED + idx)
    alpha_scored = score_with_model(model, columns, alpha_features, spec)
    selected_gate, sweep = select_gate(alpha, alpha_scored, spec)
    sweep["variant_id"] = spec.variant_id
    all_sweeps.append(sweep)

    beta_scored = score_with_model(model, columns, beta_features, spec)
    scored_cache[spec.variant_id] = beta_scored

    alpha_metrics = metric_rows(alpha, alpha_scored, "Alpha", f"{spec.variant_id}_alpha_in_sample_exploratory", "candidate_score", selected_gate)
    beta_metrics = metric_rows(beta, beta_scored, "Beta", f"{spec.variant_id}_beta_transfer", "candidate_score", selected_gate)
    for _, row in pd.concat([alpha_metrics, beta_metrics], ignore_index=True).iterrows():
        variant_metric_rows.append({"variant_id": spec.variant_id, "description": spec.description, "selected_gate": json.dumps(selected_gate), **row.to_dict()})

    beta_decoded, _ = decode_days(beta, beta_scored, "candidate_score", selected_gate)
    for site in sorted(beta["substation_id"].unique()):
        site_metrics = metric_rows(beta.loc[beta["substation_id"].eq(site)], beta_scored.loc[beta_scored["substation_id"].eq(site)], "Beta", f"{spec.variant_id}_beta_site_{site}", "candidate_score", selected_gate)
        for _, row in site_metrics.iterrows():
            beta_site_rows.append({"variant_id": spec.variant_id, **row.to_dict()})

    beta_decoded["confusion_group"] = np.select(
        [beta_decoded["label_day"] & beta_decoded["pred_day"], ~beta_decoded["label_day"] & beta_decoded["pred_day"], beta_decoded["label_day"] & ~beta_decoded["pred_day"]],
        ["TP", "FP", "FN"],
        default="TN",
    )
    summary = beta_decoded.groupby(["substation_id", "confusion_group"], as_index=False).size()
    for _, row in summary.iterrows():
        error_summary_rows.append({"variant_id": spec.variant_id, **row.to_dict()})

    beta_day = beta_metrics.loc[beta_metrics["level"].eq("day")].iloc[0]
    variant_detail_rows.append({
        "variant_id": spec.variant_id,
        "description": spec.description,
        "candidate_set": spec.candidate_set,
        "feature_mode": spec.feature_mode,
        "selection_mode": spec.selection_mode,
        "gate_mode": spec.gate_mode,
        "rerank_weight": spec.rerank_weight,
        "hard_negative_mode": spec.hard_negative_mode,
        "selection_scope": "alpha_in_sample_exploratory",
        "beta_day_precision": beta_day["precision"],
        "beta_day_recall": beta_day["recall"],
        "beta_day_f1": beta_day["f1"],
        "selected_gate": json.dumps(selected_gate),
    })

variant_metrics = pd.DataFrame(variant_metric_rows)
variant_details = pd.DataFrame(variant_detail_rows)
beta_site_metrics = pd.DataFrame(beta_site_rows)
error_summary = pd.DataFrame(error_summary_rows)
gate_sweeps = pd.concat(all_sweeps, ignore_index=True)

leaderboard = variant_details.sort_values(["beta_day_f1", "beta_day_precision", "beta_day_recall"], ascending=[False, False, False]).reset_index(drop=True)
leaderboard.insert(0, "rank", np.arange(1, len(leaderboard) + 1))
best_variant = leaderboard.iloc[0]["variant_id"]
best_spec = next(spec for spec in variant_specs if spec.variant_id == best_variant)
best_gate = json.loads(leaderboard.iloc[0]["selected_gate"])
best_scored = scored_cache[best_variant]
best_decoded, _ = decode_days(beta, best_scored, "candidate_score", best_gate)

write_csv(variant_metrics, METRICS_DIR / "03_variant_metrics_long.csv")
write_csv(leaderboard, METRICS_DIR / "02_variant_leaderboard.csv")
write_csv(variant_details, METRICS_DIR / "03_variant_details.csv")
write_csv(beta_site_metrics, METRICS_DIR / "04_beta_site_metrics_by_variant.csv")
write_csv(error_summary, METRICS_DIR / "05_beta_error_summary_by_variant.csv")
write_csv(best_decoded, INTERMEDIATE_DIR / "06_best_variant_predictions.csv")
write_csv(gate_sweeps, INTERMEDIATE_DIR / "08_alpha_gate_sweeps_by_variant.csv")

display(leaderboard[["rank", "variant_id", "beta_day_precision", "beta_day_recall", "beta_day_f1", "candidate_set", "gate_mode"]])


## 7. Failure Examples, Figures, And Manifest

The final artifacts compare baselines, variants, sites, and pseudo-load feature distributions. The HTML examples include raw net load, solar, pseudo-load, manual labels, and the best variant prediction.

In [ ]:
def confusion_label(df: pd.DataFrame) -> pd.Series:
    return pd.Series(np.select([df["label_day"] & df["pred_day"], ~df["label_day"] & df["pred_day"], df["label_day"] & ~df["pred_day"]], ["TP", "FP", "FN"], default="TN"), index=df.index)


best_decoded["confusion_group"] = confusion_label(best_decoded)
examples = (
    best_decoded.sort_values(["confusion_group", "candidate_score"], ascending=[True, False])
    .groupby("confusion_group", as_index=False)
    .head(8)
)
write_csv(examples, INTERMEDIATE_DIR / "07_best_variant_examples.csv")


def save_leaderboard_figure(leaderboard: pd.DataFrame, path: Path) -> Path:
    plot = leaderboard.sort_values("beta_day_f1", ascending=True)
    y = np.arange(len(plot))
    fig, ax = plt.subplots(figsize=(8.0, 5.5))
    ax.barh(y - 0.22, plot["beta_day_precision"], height=0.2, label="Precision", color=PALETTE["dark_blue"])
    ax.barh(y, plot["beta_day_recall"], height=0.2, label="Recall", color=PALETTE["orange"])
    ax.barh(y + 0.22, plot["beta_day_f1"], height=0.2, label="F1", color=PALETTE["grey"])
    ax.set_yticks(y)
    ax.set_yticklabels(plot["variant_id"], fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_axisbelow(True)
    ax.grid(axis="x", color=PALETTE["light_white"], linewidth=0.8)
    ax.set_title("Beta day metrics by pseudo-load variant")
    ax.legend(frameon=False, ncol=3)
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return path


def save_site_heatmap(site_metrics: pd.DataFrame, path: Path) -> Path:
    day = site_metrics.loc[site_metrics["level"].eq("day")].copy()
    day["site"] = day["fold_id"].str.extract(r"beta_site_(beta_[A-H])")[0]
    pivot = day.pivot_table(index="variant_id", columns="site", values="f1", aggfunc="first").reindex(leaderboard["variant_id"])
    fig, ax = plt.subplots(figsize=(8.0, 5.8))
    im = ax.imshow(pivot.to_numpy(), aspect="auto", cmap="YlOrBr", vmin=0, vmax=1)
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45)
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    ax.set_title("Beta site day F1 by variant")
    fig.colorbar(im, ax=ax, label="Day F1")
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return path


def save_feature_boxplot(scored: pd.DataFrame, decoded: pd.DataFrame, path: Path) -> Path:
    joined = decoded[["substation_id", "date", "confusion_group"]].merge(scored, on=["substation_id", "date"], how="left")
    features = ["pseudo_load_std_inside", "pseudo_load_roughness_inside", "pseudo_roughness_vs_daytime", "pseudo_stability_score"]
    groups = ["TP", "FP", "FN", "TN"]
    fig, axes = plt.subplots(2, 2, figsize=(9.0, 6.8))
    for ax, feature in zip(axes.ravel(), features):
        data = [joined.loc[joined["confusion_group"].eq(g), feature].dropna().to_numpy() for g in groups]
        ax.boxplot(data, tick_labels=groups, patch_artist=True, boxprops={"facecolor": PALETTE["light_white"], "color": PALETTE["dark_blue"]}, medianprops={"color": PALETTE["orange"]})
        ax.set_title(feature, fontsize=10)
        ax.grid(axis="y", color=PALETTE["light_white"], linewidth=0.8)
    fig.suptitle("Pseudo-load feature distributions by Beta confusion group", fontsize=13)
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return path


def save_confusion_comparison(path: Path) -> Path:
    rows = []
    for label, table in [("current m9", baseline_metrics), ("best pseudo", variant_metrics.loc[variant_metrics["variant_id"].eq(best_variant)])]:
        day = table.loc[(table["dataset"].eq("Beta")) & (table["level"].eq("day"))].iloc[0]
        rows.append((label, np.array([[day["tp"], day["fn"]], [day["fp"], day["tn"]]], dtype=float)))
    fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.5))
    for ax, (label, cm) in zip(axes, rows):
        im = ax.imshow(cm, cmap="YlOrBr")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred RPF", "Pred no"])
        ax.set_yticks([0, 1]); ax.set_yticklabels(["Actual RPF", "Actual no"])
        ax.set_title(label)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{int(cm[i, j])}", ha="center", va="center", color=PALETTE["dark_blue"])
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8)
    fig.tight_layout()
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return path


def save_example_html(site: str, date: str, path: Path) -> Path:
    day = beta.loc[beta["substation_id"].eq(site) & beta["date"].eq(date)].sort_values("timestamp")
    dec = best_decoded.loc[best_decoded["substation_id"].eq(site) & best_decoded["date"].eq(date)].iloc[0]
    pseudo = day["solar_MW"] - day["net_load_MW"]
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Scatter(x=day["timestamp"], y=day["net_load_MW"], mode="lines", name="Raw net load", line=dict(color=PALETTE["dark_blue"])), secondary_y=False)
    fig.add_trace(go.Scatter(x=day["timestamp"], y=day["solar_MW"], mode="lines", name="Solar", line=dict(color=PALETTE["orange"])), secondary_y=True)
    fig.add_trace(go.Scatter(x=day["timestamp"], y=pseudo, mode="lines", name="Pseudo-load", line=dict(color=PALETTE["grey"], dash="dot")), secondary_y=False)
    labelled = day.loc[day["label_interval"]]
    if not labelled.empty:
        fig.add_vrect(x0=labelled["timestamp"].iloc[0], x1=labelled["timestamp"].iloc[-1], fillcolor="rgba(235,147,44,0.18)", line_width=0, annotation_text="manual")
    if bool(dec["pred_day"]):
        fig.add_vrect(x0=dec["pred_start"], x1=dec["pred_end"], fillcolor="rgba(47,77,103,0.18)", line_width=0, annotation_text="best")
    fig.update_layout(title=f"{best_variant} | {site} | {date} | {dec['confusion_group']}", template="plotly_white", font=dict(family="Arial", color=PALETTE["dark_blue"]), height=520)
    fig.update_yaxes(title_text="MW / pseudo-load", secondary_y=False)
    fig.update_yaxes(title_text="Solar (MW)", secondary_y=True)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(path)
    return path


fig1 = save_leaderboard_figure(leaderboard, FIGURES_DIR / "fig01_beta_day_metrics_by_variant.png")
fig2 = save_site_heatmap(beta_site_metrics, FIGURES_DIR / "fig02_beta_site_f1_heatmap.png")
fig3 = save_feature_boxplot(best_scored, best_decoded, FIGURES_DIR / "fig03_pseudoload_feature_distributions.png")
fig4 = save_confusion_comparison(FIGURES_DIR / "fig04_confusion_comparison.png")

html_files = []
for _, row in examples.iterrows():
    name = f"{row['confusion_group']}_{row['substation_id']}_{row['date']}.html"
    html_files.append(save_example_html(str(row["substation_id"]), str(row["date"]), HTML_DIR / name).name)

manifest = {
    "mode": "m9_pseudoload_iterative_search",
    "beta_guided_exploratory": True,
    "publication_ready": False,
    "elapsed_seconds": time.perf_counter() - t0,
    "variant_count": int(len(variant_specs)),
    "best_variant": best_variant,
    "best_beta_day_f1": float(leaderboard.iloc[0]["beta_day_f1"]),
    "best_beta_day_precision": float(leaderboard.iloc[0]["beta_day_precision"]),
    "best_beta_day_recall": float(leaderboard.iloc[0]["beta_day_recall"]),
    "outputs": {
        "metrics": ["01_baseline_metrics.csv", "02_variant_leaderboard.csv", "03_variant_details.csv", "04_beta_site_metrics_by_variant.csv", "05_beta_error_summary_by_variant.csv"],
        "figures": [fig1.name, fig2.name, fig3.name, fig4.name],
        "html": html_files,
    },
}
with (MANIFEST_DIR / "run_manifest.json").open("w", encoding="utf-8") as fh:
    json.dump(manifest, fh, indent=2, default=str)

display(pd.DataFrame([manifest]))
